This notebook has been adapted from [source](https://github.com/dair-ai/pytorch_notebooks)

## Emototion Detection

In [ ]:
import torch
import pickle
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import time

```json
@inproceedings{saravia-etal-2018-carer,
    title = "{CARER}: Contextualized Affect Representations for Emotion Recognition",
    author = "Saravia, Elvis  and
      Liu, Hsien-Chi Toby  and
      Huang, Yen-Hao  and
      Wu, Junlin  and
      Chen, Yi-Shin",
    booktitle = "Proceedings of the 2018 Conference on Empirical Methods in Natural Language Processing"
    year = "2018",
    address = "Brussels, Belgium",
    publisher = "Association for Computational Linguistics",
    url = "https://www.aclweb.org/anthology/D18-1404",
    doi = "10.18653/v1/D18-1404",
    pages = "3687--3697"
}
```

In [ ]:
!gdown "1OB-GpME7CyFPHan9W_9Njtc2_PmvvKK7"

In [ ]:
import pandas as pd
data = pd.read_csv("emotion_detection_dataset.csv")
data

In [ ]:
data['emotions'].value_counts().plot(kind="bar")

## Tokenizing and Sampling
In the next steps we are going to **tokenize** our pieces of text, create index mapping for words, and also construct a vocabulary.

In [ ]:
## retain only text that contain less that 70 tokens to avoid too much padding
data["token_size"] = data["text"].apply(lambda x: len(x.split(' ')))
data = data.loc[data['token_size'] < 70].copy()

## sampling; we don't need to use the entire data as we only want to show you
## the tokenization/batch preparation process
data = data.sample(n=50000);

print(data.head(10))

### Building Vocabulary
After tokenizing text, it's time to build the vocabulary, which is used to determine the features that we will be using to train the models.

The code below takes create of creating the vocabulary. The code you see below is standard code you will see in many tutorials. Take some time to understand the intuition behind it.


In [ ]:
## This class creates a word -> index mapping (e.g,. "dad" -> 5) and vice-versa
## (e.g., 5 -> "dad") for the dataset
class ConstructVocab():
    def __init__(self, sentences):
        self.sentences = sentences
        self.word2idx = {}
        self.idx2word = {}
        self.vocab = set()
        self.create_index()

    def create_index(self):
        for s in self.sentences:
            # update with individual tokens
            self.vocab.update(s.split(' '))

        # sort the vocab
        self.vocab = sorted(self.vocab)

        # add a padding token with index 0
        self.word2idx['<pad>'] = 0

        # word to index mapping
        for index, word in enumerate(self.vocab):
            self.word2idx[word] = index + 1 # +1 because of pad token

        # index to word mapping
        for word, index in self.word2idx.items():
            self.idx2word[index] = word

In [ ]:
## construct vocab and indexing
inputs = ConstructVocab(data["text"].values.tolist())

## examples of what is in the vocab
inputs.vocab[0:10]

In [ ]:
## obtain id of token
inputs.word2idx['a']

In [ ]:
## you can do the reverse now -- obtain the token via the id
inputs.idx2word[1]

### Converting Data Into Tensors
Now that we have created our vocab, we can now try to convert the text input into a tensor format, which inolves a vectorization processing using those vocab ids defined above.

Let's try:

In [ ]:
## vectorize to tensor
input_tensor = [[inputs.word2idx[s] for s in es.split(' ')]  for es in data["text"].values.tolist()]

In [ ]:
## examples of what is in the input tensors
input_tensor[0:2]

### Padding Data
In order to train our recurrent neural network (RNN) model for text classification (prsented in the next notebook), we require some type of padding to generate inputs of same length. RNNs expect inputs of same lenght.

In [ ]:
## function to find max lenght of batch
def max_length(tensor):
    return max(len(t) for t in tensor)

In [ ]:
## calculate the max_length of input tensor
max_length_inp = max_length(input_tensor)
print(max_length_inp)

In [ ]:
## padding sequences

import numpy as np

def pad_sequences(x, max_len):
    padded = np.zeros((max_len), dtype=np.int64)
    if len(x) > max_len: padded[:] = x[:max_len]
    else: padded[:len(x)] = x
    return padded

In [ ]:
## inplace padding
input_tensor = [pad_sequences(x, max_length_inp) for x in input_tensor]

### Binarization
We would like to binarize our `target` values in our dataset so that we can obtain `one-hot encodings` for the target values.

In [ ]:
import time
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
import itertools
import pandas as pd
from scipy import stats
from sklearn import metrics
from sklearn.preprocessing import LabelEncoder

## convert targets to one-hot encoding vectors
emotions = list(set(data.emotions.unique()))
num_emotions = len(emotions)

## binarizer
mlb = preprocessing.MultiLabelBinarizer()
data_labels =  [set(emos) & set(emotions) for emos in data[['emotions']].values]
bin_emotions = mlb.fit_transform(data_labels)
target_tensor = np.array(bin_emotions.tolist())

In [ ]:
## print the one-hot encodings for the first two examples in the dataset
target_tensor[0:2]

In [ ]:
## we can easily check the original values for the first two piece of text
data[0:2]

In [ ]:
## we can easily find the emotion index for instance of the first example above: fear
get_emotion = lambda t: np.argmax(t)
get_emotion(target_tensor[0])

In [ ]:
## pass that index to a key/value object to obtain the original emotion
emotion_dict = {0: 'anger', 1: 'fear', 2: 'joy', 3: 'love', 4: 'sadness', 5: 'surprise'}
emotion_dict[get_emotion(target_tensor[0])]

### Split Data

We would like to split our data into a train and validation set. In addition, we also want a holdout dataset (test set) for evaluating the models.

Let's do that below. We are going to use the built in `train_test_split` dataset from scikit learn. Let's do that below:

In [ ]:
## Creating training and validation sets using an 80-20 split
input_tensor_train, input_tensor_val, target_tensor_train, target_tensor_val = train_test_split(input_tensor, target_tensor, test_size=0.2)

## Split the validataion further to obtain a holdout dataset (for testing) -- split 50:50
input_tensor_val, input_tensor_test, target_tensor_val, target_tensor_test = train_test_split(input_tensor_val, target_tensor_val, test_size=0.5)

## Show length
len(input_tensor_train), len(target_tensor_train), len(input_tensor_val), len(target_tensor_val), len(input_tensor_test), len(target_tensor_test)

### Dataset and Data Loader

We can also load the data into a PyTorch `DataLoader` object, which makes it easy to manipulate the data, create batches, and apply further transformations. This simplified our training procedure as well -- we will look a this in the next notebook.

In [ ]:
## Define a few useful parameters

TRAIN_BUFFER_SIZE = len(input_tensor_train)
VAL_BUFFER_SIZE = len(input_tensor_val)
TEST_BUFFER_SIZE = len(input_tensor_test)
BATCH_SIZE = 64

TRAIN_N_BATCH = TRAIN_BUFFER_SIZE // BATCH_SIZE
VAL_N_BATCH = VAL_BUFFER_SIZE // BATCH_SIZE
TEST_N_BATCH = TEST_BUFFER_SIZE // BATCH_SIZE

In [ ]:
from torch.utils.data import Dataset, DataLoader

We use the `Dataset` class to represent a dataset object. You override a few methods and you should be able to create the dataset object you need.

In [ ]:
class MyData(Dataset):
    def __init__(self, X, y):
        self.data = X
        self.target = y
        self.length = [ np.sum(1 - np.equal(x, 0)) for x in X]

    def __getitem__(self, index):
        x = self.data[index]
        y = self.target[index]
        x_len = self.length[index]
        return x, y, x_len

    def __len__(self):
        return len(self.data)

Load the data and then pass it to an `iterator` called `DataLoadder` which finally defines how samples/batches should be prepared.

In [ ]:
## Dataset instance
train_dataset = MyData(input_tensor_train, target_tensor_train)
val_dataset = MyData(input_tensor_val, target_tensor_val)
test_dataset = MyData(input_tensor_test, target_tensor_test)


## Data Loader instance
train_dataset = DataLoader(train_dataset, batch_size = BATCH_SIZE,
                     drop_last=True,
                     shuffle=True)

val_dataset = DataLoader(val_dataset, batch_size = BATCH_SIZE,
                     drop_last=True,
                     shuffle=True)

test_dataset = DataLoader(test_dataset, batch_size = BATCH_SIZE,
                     drop_last=True,
                     shuffle=True)

In [ ]:
val_dataset.batch_size

In [ ]:
## preview of the data
val_dataset.dataset.data[0:2]

## RNN

Let's make a simple RNN model with 1 RNN layer and a fully connected layer. We will also need an embedding layer to embed the input sequence to a lower dimension and then a dropout layer to reduce overfitting.

In [ ]:
class EmoRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_units, batch_sz, output_size):
        super(EmoRNN, self).__init__()
        self.batch_sz = batch_sz
        self.hidden_units = hidden_units
        self.embedding_dim = embedding_dim
        self.vocab_size = vocab_size
        self.output_size = output_size

        ## layers
        self.embedding = nn.Embedding(self.vocab_size, self.embedding_dim)
        self.dropout = nn.Dropout(p=0.5) # avoid overfitting
        self.rnn = nn.RNN(self.embedding_dim, self.hidden_units)
        self.fc = nn.Linear(self.hidden_units, self.output_size)

    def initialize_hidden_state(self):
        return torch.zeros(1, self.batch_sz, self.hidden_units)

    def forward(self, x, lens):
        x = self.embedding(x)
        self.hidden = self.initialize_hidden_state().to(x.device)
        output, self.hidden = self.rnn(x, self.hidden) # max_len X batch_size X hidden_units
        out = output[-1, :, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out, self.hidden

In [ ]:
# parameters
TRAIN_BUFFER_SIZE = 40000 # len(input_tensor_train)
VAL_BUFFER_SIZE = 5000 # len(input_tensor_val)
TEST_BUFFER_SIZE = 5000 # len(input_tensor_test)
BATCH_SIZE = 64
TRAIN_N_BATCH = TRAIN_BUFFER_SIZE // BATCH_SIZE
VAL_N_BATCH = VAL_BUFFER_SIZE // BATCH_SIZE
TEST_N_BATCH = TEST_BUFFER_SIZE // BATCH_SIZE

embedding_dim = 256
units = 1024
vocab_inp_size = 27291 # len(inputs.word2idx)
target_size = 6 # num_emotions

In [ ]:
# Device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
## Enabling cuda
model = EmoRNN(vocab_inp_size, embedding_dim, units, BATCH_SIZE, target_size)
model.to(device)

## loss criterion and optimizer for training
criterion = nn.CrossEntropyLoss() # the same as log_softmax + NLLLoss
optimizer = torch.optim.Adam(model.parameters())

In [ ]:
def loss_function(y, prediction):
    """ CrossEntropyLoss expects outputs and class indices as target """
    ## convert from one-hot encoding to class indices
    target = torch.max(y, 1)[1]
    loss = criterion(prediction, target)
    return loss

def accuracy(target, logit):
    ''' Obtain accuracy for training round '''
    target = torch.max(target, 1)[1] # convert from one-hot encoding to class indices
    corrects = (torch.max(logit, 1)[1].data == target).sum()
    accuracy = 100.0 * corrects / len(logit)
    return accuracy

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):
    start = time.time()

    ### Initialize hidden state
    # TODO: do initialization here.
    total_loss = 0
    train_accuracy, val_accuracy = 0, 0

    ### Training
    for (batch, (inp, targ, lens)) in enumerate(train_dataset):
        loss = 0
        inp = inp.permute(1 ,0).to(device)
        predictions, _ = model(inp, lens)

        loss += loss_function(targ.to(device), predictions)
        batch_loss = (loss / int(targ.shape[1]))
        total_loss += batch_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_accuracy = accuracy(targ.to(device), predictions)
        train_accuracy += batch_accuracy

        if batch % 100 == 0:
            print('Epoch {} Batch {} Val. Loss {:.4f}'.format(epoch + 1,
                                                         batch,
                                                         batch_loss.cpu().detach().numpy()))

    ### Validating
    for (batch, (inp, targ, lens)) in enumerate(val_dataset):
        predictions,_ = model(inp.permute(1, 0).to(device), lens)
        batch_accuracy = accuracy(targ.to(device), predictions)
        val_accuracy += batch_accuracy

    print('Epoch {} Loss {:.4f} -- Train Acc. {:.4f} -- Val Acc. {:.4f}'.format(epoch + 1,
                                                             total_loss / TRAIN_N_BATCH,
                                                             train_accuracy / TRAIN_N_BATCH,
                                                             val_accuracy / VAL_N_BATCH))
    print('Time taken for 1 epoch {} sec\n'.format(time.time() - start))

## GRU

We will just replace the RNN layer with GRU layer.

In [ ]:
class EmoGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_units, batch_sz, output_size):
        super().__init__()
        self.batch_sz = batch_sz
        self.hidden_units = hidden_units
        self.embedding_dim = embedding_dim
        self.vocab_size = vocab_size
        self.output_size = output_size

        ## layers
        self.embedding = nn.Embedding(self.vocab_size, self.embedding_dim)
        self.dropout = nn.Dropout(p=0.5) # avoid overfitting
        self.rnn = nn.GRU(self.embedding_dim, self.hidden_units)
        self.fc = nn.Linear(self.hidden_units, self.output_size)

    def initialize_hidden_state(self):
        return torch.zeros(1, self.batch_sz, self.hidden_units)

    def forward(self, x, lens):
        x = self.embedding(x)
        self.hidden = self.initialize_hidden_state().to(x.device)
        output, self.hidden = self.rnn(x, self.hidden) # max_len X batch_size X hidden_units
        out = output[-1, :, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out, self.hidden

In [ ]:
## Enabling cuda
model = EmoGRU(vocab_inp_size, embedding_dim, units, BATCH_SIZE, target_size)
model.to(device)

## loss criterion and optimizer for training
criterion = nn.CrossEntropyLoss() # the same as log_softmax + NLLLoss
optimizer = torch.optim.Adam(model.parameters())

In [ ]:
EPOCHS = 3

for epoch in range(EPOCHS):
    start = time.time()

    ### Initialize hidden state
    # TODO: do initialization here.
    total_loss = 0
    train_accuracy, val_accuracy = 0, 0

    ### Training
    for (batch, (inp, targ, lens)) in enumerate(train_dataset):
        loss = 0
        inp = inp.permute(1 ,0).to(device)
        predictions, _ = model(inp, lens)

        loss += loss_function(targ.to(device), predictions)
        batch_loss = (loss / int(targ.shape[1]))
        total_loss += batch_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_accuracy = accuracy(targ.to(device), predictions)
        train_accuracy += batch_accuracy

        if batch % 100 == 0:
            print('Epoch {} Batch {} Val. Loss {:.4f}'.format(epoch + 1,
                                                         batch,
                                                         batch_loss.cpu().detach().numpy()))

    ### Validating
    for (batch, (inp, targ, lens)) in enumerate(val_dataset):
        predictions,_ = model(inp.permute(1, 0).to(device), lens)
        batch_accuracy = accuracy(targ.to(device), predictions)
        val_accuracy += batch_accuracy

    print('Epoch {} Loss {:.4f} -- Train Acc. {:.4f} -- Val Acc. {:.4f}'.format(epoch + 1,
                                                             total_loss / TRAIN_N_BATCH,
                                                             train_accuracy / TRAIN_N_BATCH,
                                                             val_accuracy / VAL_N_BATCH))
    print('Time taken for 1 epoch {} sec\n'.format(time.time() - start))

## LSTM

Now, we will replace the RNN layer with LSTM layer. Remember LSTM also keeps a cell state so we need to add that too.

In [ ]:
class EmoLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_units, batch_sz, output_size):
        super().__init__()
        self.batch_sz = batch_sz
        self.hidden_units = hidden_units
        self.embedding_dim = embedding_dim
        self.vocab_size = vocab_size
        self.output_size = output_size

        ## layers
        self.embedding = nn.Embedding(self.vocab_size, self.embedding_dim)
        self.dropout = nn.Dropout(p=0.5) # avoid overfitting
        self.rnn = nn.LSTM(self.embedding_dim, self.hidden_units)
        self.fc = nn.Linear(self.hidden_units, self.output_size)

    def initialize_hidden_state(self):
        return torch.zeros(1, self.batch_sz, self.hidden_units)

    def forward(self, x, lens):
        x = self.embedding(x)
        self.hidden = self.initialize_hidden_state().to(x.device)
        self.cell_state = self.initialize_hidden_state().to(x.device)
        output, (self.hidden, self.cell_state) = self.rnn(x, (self.hidden, self.cell_state)) # max_len X batch_size X hidden_units
        out = output[-1, :, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out, self.hidden

In [ ]:
## Enabling cuda
model = EmoLSTM(vocab_inp_size, embedding_dim, units, BATCH_SIZE, target_size)
model.to(device)

## loss criterion and optimizer for training
criterion = nn.CrossEntropyLoss() # the same as log_softmax + NLLLoss
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):
    start = time.time()

    ### Initialize hidden state
    # TODO: do initialization here.
    total_loss = 0
    train_accuracy, val_accuracy = 0, 0

    ### Training
    for (batch, (inp, targ, lens)) in enumerate(train_dataset):
        loss = 0
        inp = inp.permute(1 ,0).to(device)
        predictions, _ = model(inp, lens)

        loss += loss_function(targ.to(device), predictions)
        batch_loss = (loss / int(targ.shape[1]))
        total_loss += batch_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_accuracy = accuracy(targ.to(device), predictions)
        train_accuracy += batch_accuracy

        if batch % 100 == 0:
            print('Epoch {} Batch {} Val. Loss {:.4f}'.format(epoch + 1,
                                                         batch,
                                                         batch_loss.cpu().detach().numpy()))

    ### Validating
    for (batch, (inp, targ, lens)) in enumerate(val_dataset):
        predictions,_ = model(inp.permute(1, 0).to(device), lens)
        batch_accuracy = accuracy(targ.to(device), predictions)
        val_accuracy += batch_accuracy

    print('Epoch {} Loss {:.4f} -- Train Acc. {:.4f} -- Val Acc. {:.4f}'.format(epoch + 1,
                                                             total_loss / TRAIN_N_BATCH,
                                                             train_accuracy / TRAIN_N_BATCH,
                                                             val_accuracy / VAL_N_BATCH))
    print('Time taken for 1 epoch {} sec\n'.format(time.time() - start))